In [3]:
!pip install sentence-transformers

import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Загрузка датасетов
with open('code_corpus.json', 'r', encoding='utf-8') as f:
    code_corpus = json.load(f)
print("Количество фрагментов кода:", len(code_corpus))

with open('eval_questions.json', 'r', encoding='utf-8') as f:
    eval_questions = json.load(f)
print("Количество вопросов:", len(eval_questions))

# Загрузка моделей эмбеддингов
print("Загрузка модели 1")
model_1 = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Загрузка модели 2")
model_2 = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

print("Все модели загружены")


Количество фрагментов кода: 200
Количество вопросов: 25
Загрузка модели 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Загрузка модели 2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Все модели загружены


In [6]:
from sentence_transformers import util

# Достал чистый текст кода
code_texts = [item['code'] for item in code_corpus]

# Достал текст вопросов
query_texts = [item['question'] if 'question' in item else item['query'] if 'query' in item else item['text'] for item in eval_questions]

# Сделал векторы для модели 1
print("Генерация векторов для модели 1")
corpus_embeddings_1 = model_1.encode(code_texts, convert_to_tensor=True)
queries_embeddings_1 = model_1.encode(query_texts, convert_to_tensor=True)

# Сделал векторы для модели 2
print("Генерация векторов для модели 2")
corpus_embeddings_2 = model_2.encode(code_texts, convert_to_tensor=True)
queries_embeddings_2 = model_2.encode(query_texts, convert_to_tensor=True)

# Функция поиска топ 3 похожих вариантов
def find_top_3(query_embedding, corpus_embeddings):
    cos_scores = util.cos_sim(query_embedding, corpus_embeddings)
    # Превращаем в обычную строку чисел
    scores_np = cos_scores.cpu().numpy()[0]
    # Находим индексы 3 самых больших значений
    top_results = np.argsort(-scores_np)[:3]
    return [(idx, scores_np[idx]) for idx in top_results]

# Проверка функции на первом вопросе
first_query_results = find_top_3(queries_embeddings_1[0], corpus_embeddings_1)
print("Индексы топ-3 ответов для вопроса 0:", first_query_results)


Генерация векторов для модели 1
Генерация векторов для модели 2
Индексы топ-3 ответов для вопроса 0: [(np.int64(170), np.float32(0.36320543)), (np.int64(70), np.float32(0.34233758)), (np.int64(71), np.float32(0.338295))]


In [9]:
# Запомнил текстовые ID из базы кода по порядку
corpus_ids = [item['id'] for item in code_corpus]

# Функция для подсчета точности Hit@1 и Hit@3
def evaluate_model(queries_embeddings, corpus_embeddings):
    hit_1 = 0
    hit_3 = 0
    total = len(eval_questions)

    for i, query in enumerate(eval_questions):
        # Достал правильный ID ответа из вопроса
        true_id = query['correct_chunk_id']

        # Считаю косинусное сходство для этого вопроса
        cos_scores = util.cos_sim(queries_embeddings[i], corpus_embeddings)
        scores_np = cos_scores.cpu().numpy().flatten()  # Выпрямил массив в одну строку
        top_results = np.argsort(-scores_np)[:3]

        # Перевел порядковые номера в текстовые ID
        top_3_ids = [corpus_ids[idx] for idx in top_results]

        # Проверяю первое место для Hit@1
        if top_3_ids[0] == true_id:
            hit_1 += 1

        # Проверяем попадание в тройку для Hit@3
        if true_id in top_3_ids:
            hit_3 += 1

    return (hit_1 / total) * 100, (hit_3 / total) * 100

# Считаю точность для модели 1
hit1_m1, hit3_m1 = evaluate_model(queries_embeddings_1, corpus_embeddings_1)
print("Модель 1 - Hit@1:", hit1_m1, "%, Hit@3:", hit3_m1, "%")

# Считаю точность для модели 2
hit1_m2, hit3_m2 = evaluate_model(queries_embeddings_2, corpus_embeddings_2)
print("Модель 2 - Hit@1:", hit1_m2, "%, Hit@3:", hit3_m2, "%")


Модель 1 - Hit@1: 40.0 %, Hit@3: 68.0 %
Модель 2 - Hit@1: 28.000000000000004 %, Hit@3: 84.0 %
